# Pharmacogene ML Features - Exploratory Data Analysis
**DNA Gene Mapping Project - ML Phase V5**  
**Author:** Sharique Mohammad  
**Date:** February 2026  
**Table:** pharmacogene_ml_features (4.1M variants, 72 columns)

## Objective
Explore pharmacogenomic features: drug metabolism roles, druggability scores, pharmacogene categories, and variant impacts on drug response.

## Key Questions
1. What is the distribution of pharmacogene categories and roles?
2. How are drug metabolism roles distributed across variants?
3. What is the druggability landscape?
4. Which genes have the highest pharmacogenomic burden?
5. How do PharmGKB annotations relate to pathogenicity?

## Deliverables
- 12+ visualizations saved to pharmacogene_ml_features/images/
- EDA report saved to pharmacogene_ml_features/reports/
- Missing values, correlation matrix, feature statistics saved to pharmacogene_ml_features/metrics/

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import os
from pathlib import Path
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

load_dotenv()

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

PROJECT_ROOT = Path().absolute().parent.parent
BASE_OUT     = PROJECT_ROOT / 'data' / 'analytical' / 'pharmacogene_ml_features'
IMAGES_DIR   = BASE_OUT / 'images'
REPORTS_DIR  = BASE_OUT / 'reports'
METRICS_DIR  = BASE_OUT / 'metrics'

IMAGES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete")
print(f"Images  : {IMAGES_DIR}")
print(f"Reports : {REPORTS_DIR}")
print(f"Metrics : {METRICS_DIR}")

## 2. Database Connection

In [ ]:
POSTGRES_HOST     = os.getenv("POSTGRES_HOST")
POSTGRES_PORT     = os.getenv("POSTGRES_PORT")
POSTGRES_DB       = os.getenv("POSTGRES_DB")
POSTGRES_USER     = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")

conn_str = f"postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
engine = create_engine(conn_str)

print("Database connection established")
print(f"Host : {POSTGRES_HOST}:{POSTGRES_PORT}")
print(f"DB   : {POSTGRES_DB}")

## 3. Data Loading

In [ ]:
print("Loading pharmacogene_ml_features (10% sample)...")
query = """
    SELECT * FROM gold.pharmacogene_ml_features
    TABLESAMPLE SYSTEM (10)
"""
df = pd.read_sql(query, engine)

print(f"Rows loaded    : {len(df):,}")
print(f"Columns        : {len(df.columns)}")
print(f"Memory usage   : {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"Full table est : ~4.1M rows")

## 4. Type Conversion

In [ ]:
# INT columns from gold schema
int_cols = [
    'mutation_severity_score', 'pathogenicity_score', 'pharmgkb_source_count',
    'gene_pharmacogene_variants', 'gene_drug_interaction_variants',
    'gene_metabolizer_variants', 'gene_transporter_variants',
    'gene_pharmacogene_pathogenic', 'tissues_expressed_count',
    'cancer_mutation_count', 'disease_count'
]

# DOUBLE columns from gold schema
double_cols = [
    'druggability_score', 'enhanced_druggability_score',
    'gene_avg_druggability', 'allele_frequency'
]

# BOOLEAN columns from gold schema
bool_cols = [
    'gene_is_validated', 'gene_description_mentions_drug',
    'is_pathogenic', 'is_benign', 'is_vus',
    'is_missense_variant', 'is_loss_of_function',
    'is_pharmacogene', 'is_drug_target', 'is_metabolizing_enzyme',
    'is_enzyme', 'is_drug_transporter', 'is_kinase', 'is_phosphatase',
    'is_receptor', 'is_gpcr', 'is_transporter',
    'is_metabolizer_variant', 'is_transporter_variant',
    'is_kinase_inhibitor_target', 'has_pharmgkb_annotation',
    'gene_has_multiple_drug_variants', 'is_liver_expressed',
    'is_kidney_expressed', 'is_oncology_target', 'is_cancer_drug_target',
    'is_common_variant', 'is_rare_variant',
    'has_cancer_disease', 'has_cardiovascular_disease', 'has_neurological_disease'
]

for col in int_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

for col in double_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower().map({'true': True, 'false': False})

print("Type conversion complete")
print(df.dtypes.value_counts())

## 5. Dataset Overview

In [ ]:
total = len(df)

print("Dataset Overview")
print("=" * 60)
print(f"Total variants  : {total:,}")
print(f"Total columns   : {len(df.columns)}")
print(f"Unique genes    : {df['gene_name'].nunique():,}")
print(f"Chromosomes     : {df['chromosome'].nunique()}")
print()
print("Column list:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col:<50} {str(df[col].dtype)}")

## 6. Missing Values Analysis

In [ ]:
missing = pd.DataFrame({
    'column': df.columns,
    'missing_count': df.isnull().sum().values,
    'missing_pct': (df.isnull().sum().values / len(df) * 100).round(2)
}).sort_values('missing_pct', ascending=False)

missing_with_nulls = missing[missing['missing_count'] > 0]
print(f"Columns with missing values: {len(missing_with_nulls)}")
print()
print(missing_with_nulls.to_string(index=False))

missing.to_csv(METRICS_DIR / 'missing_values.csv', index=False)
print(f"\nSaved: {METRICS_DIR / 'missing_values.csv'}")

## 7. Target Variable Analysis

In [ ]:
pathogenic_count = int(df['is_pathogenic'].sum()) if 'is_pathogenic' in df.columns else 0
benign_count     = int(df['is_benign'].sum())     if 'is_benign'     in df.columns else 0
vus_count        = int(df['is_vus'].sum())        if 'is_vus'        in df.columns else 0

print("Target Variable Distribution")
print("=" * 50)
print(f"Pathogenic : {pathogenic_count:>10,}  ({pathogenic_count/total*100:>5.1f}%)")
print(f"Benign     : {benign_count:>10,}  ({benign_count/total*100:>5.1f}%)")
print(f"VUS        : {vus_count:>10,}  ({vus_count/total*100:>5.1f}%)")

if pathogenic_count > 0 and benign_count > 0:
    imbalance_ratio = max(pathogenic_count, benign_count) / min(pathogenic_count, benign_count)
    print(f"\nClass imbalance ratio : {imbalance_ratio:.2f}:1")
    print(f"SMOTE needed          : {imbalance_ratio > 5}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

categories = ['Pathogenic', 'Benign', 'VUS']
counts     = [pathogenic_count, benign_count, vus_count]
colors     = ['#e74c3c', '#27ae60', '#95a5a6']

bars = axes[0].bar(categories, counts, color=colors, alpha=0.8, edgecolor='black')
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                 f'{count:,}\n({count/total*100:.1f}%)',
                 ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_ylabel('Number of Variants', fontsize=11, fontweight='bold')
axes[0].set_title('Target Variable Distribution', fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

if 'clinical_significance_simple' in df.columns:
    sig_dist = df['clinical_significance_simple'].value_counts().head(8)
    sig_dist.plot(kind='barh', ax=axes[1], color='steelblue', alpha=0.8, edgecolor='black')
    axes[1].set_xlabel('Count', fontsize=11, fontweight='bold')
    axes[1].set_title('Clinical Significance Categories', fontsize=12, fontweight='bold')
    axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(IMAGES_DIR / '01_target_variable.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 01_target_variable.png")

## 8. Pharmacogene Categories

In [ ]:
if 'pharmacogene_category' in df.columns:
    print("Pharmacogene Category Distribution")
    pg_dist = df['pharmacogene_category'].value_counts()
    print(pg_dist.to_string())

    fig, ax = plt.subplots(figsize=(12, 7))
    pg_dist.head(15).sort_values().plot(kind='barh', ax=ax, color='steelblue', alpha=0.8, edgecolor='black')
    ax.set_xlabel('Number of Variants', fontsize=11, fontweight='bold')
    ax.set_title('Pharmacogene Category Distribution', fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / '02_pharmacogene_categories.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: 02_pharmacogene_categories.png")

## 9. Drug Metabolism Roles

In [ ]:
if 'drug_metabolism_role' in df.columns:
    print("Drug Metabolism Role Distribution")
    dm_dist = df['drug_metabolism_role'].value_counts()
    print(dm_dist.to_string())

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    dm_dist.head(10).sort_values().plot(kind='barh', ax=axes[0], color='coral', alpha=0.8, edgecolor='black')
    axes[0].set_xlabel('Number of Variants', fontsize=11, fontweight='bold')
    axes[0].set_title('Drug Metabolism Role Distribution', fontsize=12, fontweight='bold')
    axes[0].grid(axis='x', alpha=0.3)

    role_flags = [
        'is_metabolizing_enzyme', 'is_drug_transporter', 'is_drug_target',
        'is_kinase', 'is_receptor', 'is_gpcr', 'is_enzyme', 'is_transporter'
    ]
    role_flags = [c for c in role_flags if c in df.columns]
    role_counts = {c.replace('is_', '').replace('_', ' ').title(): int(df[c].sum()) for c in role_flags}

    names  = list(role_counts.keys())
    values = list(role_counts.values())
    colors = plt.cm.Set2(np.linspace(0, 1, len(names)))
    bars = axes[1].barh(names, values, color=colors, alpha=0.8, edgecolor='black')
    for bar, val in zip(bars, values):
        axes[1].text(bar.get_width() + max(values)*0.01,
                     bar.get_y() + bar.get_height()/2.,
                     f'{val:,} ({val/total*100:.1f}%)', va='center', fontsize=9)
    axes[1].set_xlabel('Count', fontsize=11, fontweight='bold')
    axes[1].set_title('Protein Role Flags', fontsize=12, fontweight='bold')
    axes[1].set_xlim(0, max(values) * 1.3)
    axes[1].grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.savefig(IMAGES_DIR / '03_drug_metabolism_roles.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: 03_drug_metabolism_roles.png")

## 10. Druggability Analysis

In [ ]:
drugg_cols = ['druggability_score', 'enhanced_druggability_score', 'gene_avg_druggability']
drugg_cols = [c for c in drugg_cols if c in df.columns]

if drugg_cols:
    fig, axes = plt.subplots(1, len(drugg_cols), figsize=(6 * len(drugg_cols), 6))
    if len(drugg_cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, drugg_cols):
        data = df[col].dropna()
        ax.hist(data, bins=40, color='mediumpurple', alpha=0.8, edgecolor='black')
        median_val = data.median()
        ax.axvline(median_val, color='red', linestyle='--', linewidth=1.5,
                   label=f'Median: {median_val:.2f}')
        ax.set_title(col, fontsize=10, fontweight='bold')
        ax.set_xlabel('Score', fontsize=9)
        ax.set_ylabel('Frequency', fontsize=9)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(IMAGES_DIR / '04_druggability_scores.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: 04_druggability_scores.png")

    print("\nDruggability Score Statistics:")
    print(df[drugg_cols].describe().to_string())

## 11. PharmGKB Annotations

In [ ]:
if 'pharmgkb_evidence' in df.columns:
    print("PharmGKB Evidence Level Distribution")
    pgkb_dist = df['pharmgkb_evidence'].value_counts()
    print(pgkb_dist.to_string())

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    pgkb_dist.head(10).sort_values().plot(kind='barh', ax=axes[0], color='teal', alpha=0.8, edgecolor='black')
    axes[0].set_xlabel('Number of Variants', fontsize=11, fontweight='bold')
    axes[0].set_title('PharmGKB Evidence Level Distribution', fontsize=12, fontweight='bold')
    axes[0].grid(axis='x', alpha=0.3)

    has_annotation = int(df['has_pharmgkb_annotation'].sum()) if 'has_pharmgkb_annotation' in df.columns else 0
    no_annotation  = total - has_annotation

    axes[1].pie([has_annotation, no_annotation],
                labels=['Has PharmGKB', 'No PharmGKB'],
                colors=['#2ecc71', '#e74c3c'],
                autopct='%1.1f%%', startangle=90,
                textprops={'fontsize': 11, 'fontweight': 'bold'})
    axes[1].set_title('PharmGKB Annotation Coverage', fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.savefig(IMAGES_DIR / '05_pharmgkb_annotations.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: 05_pharmgkb_annotations.png")

## 12. Variant and Drug Response Impact

In [ ]:
if 'drug_response_impact' in df.columns:
    print("Drug Response Impact Distribution")
    dri_dist = df['drug_response_impact'].value_counts()
    print(dri_dist.to_string())

    fig, ax = plt.subplots(figsize=(12, 6))
    dri_dist.sort_values().plot(kind='barh', ax=ax, color='darkorange', alpha=0.8, edgecolor='black')
    ax.set_xlabel('Number of Variants', fontsize=11, fontweight='bold')
    ax.set_title('Drug Response Impact Distribution', fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / '06_drug_response_impact.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: 06_drug_response_impact.png")

## 13. Metabolizer Phenotype Risk

In [ ]:
if 'metabolizer_phenotype_risk' in df.columns:
    print("Metabolizer Phenotype Risk Distribution")
    mph_dist = df['metabolizer_phenotype_risk'].value_counts()
    print(mph_dist.to_string())

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    mph_dist.plot(kind='bar', ax=axes[0], color='navy', alpha=0.8, edgecolor='black')
    axes[0].set_xlabel('Metabolizer Phenotype Risk', fontsize=11, fontweight='bold')
    axes[0].set_ylabel('Number of Variants', fontsize=11, fontweight='bold')
    axes[0].set_title('Metabolizer Phenotype Risk Distribution', fontsize=12, fontweight='bold')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].grid(axis='y', alpha=0.3)

    if 'pharmgkb_source_count' in df.columns:
        axes[1].hist(df['pharmgkb_source_count'].dropna(), bins=20,
                     color='steelblue', alpha=0.8, edgecolor='black')
        axes[1].set_xlabel('PharmGKB Source Count', fontsize=11, fontweight='bold')
        axes[1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
        axes[1].set_title('PharmGKB Source Count Distribution', fontsize=12, fontweight='bold')
        axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(IMAGES_DIR / '07_metabolizer_phenotype.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: 07_metabolizer_phenotype.png")

## 14. Gene-Level Pharmacogenomics

In [ ]:
gene_pg_cols = [
    'gene_pharmacogene_variants', 'gene_drug_interaction_variants',
    'gene_metabolizer_variants', 'gene_transporter_variants',
    'gene_pharmacogene_pathogenic'
]
gene_pg_cols = [c for c in gene_pg_cols if c in df.columns]

if gene_pg_cols:
    n_cols = 3
    n_rows = (len(gene_pg_cols) + 2) // 3
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4))
    axes = axes.flatten()

    for i, col in enumerate(gene_pg_cols):
        data = df[col].dropna()
        axes[i].hist(data, bins=30, color='coral', alpha=0.8, edgecolor='black')
        axes[i].set_title(col, fontsize=10, fontweight='bold')
        axes[i].set_xlabel('Count', fontsize=9)
        axes[i].set_ylabel('Frequency', fontsize=9)
        axes[i].grid(alpha=0.3)

    for i in range(len(gene_pg_cols), len(axes)):
        axes[i].axis('off')

    plt.tight_layout()
    plt.savefig(IMAGES_DIR / '08_gene_level_pharmacogenomics.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: 08_gene_level_pharmacogenomics.png")

if 'gene_pharmacogene_priority' in df.columns:
    print("\nGene Pharmacogene Priority Distribution")
    gpp_dist = df['gene_pharmacogene_priority'].value_counts()
    print(gpp_dist.to_string())

## 15. Therapeutic Context

In [ ]:
therapeutic_flags = [
    'is_oncology_target', 'is_cancer_drug_target',
    'is_kinase_inhibitor_target', 'has_cancer_disease',
    'has_cardiovascular_disease', 'has_neurological_disease'
]
therapeutic_flags = [c for c in therapeutic_flags if c in df.columns]

flag_counts = {col: int(df[col].sum()) for col in therapeutic_flags}

print("Therapeutic Context Features")
print("=" * 50)
for col, count in sorted(flag_counts.items(), key=lambda x: -x[1]):
    print(f"  {col:<40} : {count:>10,}  ({count/total*100:.1f}%)")

fig, ax = plt.subplots(figsize=(12, 7))
names  = [c.replace('_', ' ').title() for c in therapeutic_flags]
values = [flag_counts[c] for c in therapeutic_flags]
colors = plt.cm.Set3(np.linspace(0, 1, len(names)))

bars = ax.barh(names, values, color=colors, alpha=0.8, edgecolor='black')
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + max(values)*0.01,
            bar.get_y() + bar.get_height()/2.,
            f'{val/total*100:.1f}%', va='center', fontsize=9)
ax.set_xlabel('Count', fontsize=11, fontweight='bold')
ax.set_title('Therapeutic Context Feature Rates', fontsize=12, fontweight='bold')
ax.set_xlim(0, max(values) * 1.2)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(IMAGES_DIR / '09_therapeutic_context.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 09_therapeutic_context.png")

## 16. Correlation Analysis

In [ ]:
corr_features = [
    'mutation_severity_score', 'pathogenicity_score', 'pharmgkb_source_count',
    'gene_pharmacogene_variants', 'gene_drug_interaction_variants',
    'gene_metabolizer_variants', 'gene_transporter_variants',
    'gene_pharmacogene_pathogenic', 'tissues_expressed_count',
    'cancer_mutation_count', 'disease_count',
    'druggability_score', 'enhanced_druggability_score',
    'gene_avg_druggability', 'allele_frequency'
]
corr_features = [c for c in corr_features if c in df.columns]

corr_data   = df[corr_features].apply(pd.to_numeric, errors='coerce')
corr_matrix = corr_data.corr()

corr_matrix.to_csv(METRICS_DIR / 'correlation_matrix.csv')
print(f"Saved: {METRICS_DIR / 'correlation_matrix.csv'}")

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8}, ax=ax, annot_kws={'size': 8})
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(IMAGES_DIR / '10_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 10_correlation_matrix.png")

print("\nHighly correlated pairs (|r| > 0.9):")
found = False
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        val = corr_matrix.iloc[i, j]
        if abs(val) > 0.9:
            print(f"  {corr_matrix.columns[i]} vs {corr_matrix.columns[j]}: {val:.3f}")
            found = True
if not found:
    print("  None found above 0.9")

## 17. Feature Statistics Summary

In [ ]:
stats = df[corr_features].describe().T
stats['missing_pct'] = (df[corr_features].isnull().sum() / len(df) * 100).values
stats.to_csv(METRICS_DIR / 'feature_statistics.csv')
print(f"Saved: {METRICS_DIR / 'feature_statistics.csv'}")
print()
print(stats.to_string())

## 18. EDA Report

In [ ]:
report_path      = REPORTS_DIR / 'pharmacogene_eda_report.txt'
images_generated = sorted(IMAGES_DIR.glob('*.png'))

with open(report_path, 'w') as f:
    f.write("=" * 80 + "\n")
    f.write("PHARMACOGENE ML FEATURES - EDA REPORT\n")
    f.write("DNA Gene Mapping Project - ML Phase V5\n")
    f.write("=" * 80 + "\n\n")

    f.write("TABLE: gold.pharmacogene_ml_features\n")
    f.write(f"Sample rows  : {len(df):,} (10% of ~4.1M)\n")
    f.write(f"Columns      : {len(df.columns)}\n")
    f.write(f"Unique genes : {df['gene_name'].nunique():,}\n\n")

    f.write("TARGET VARIABLE (is_pathogenic)\n")
    f.write("-" * 40 + "\n")
    f.write(f"Pathogenic : {pathogenic_count:,} ({pathogenic_count/total*100:.1f}%)\n")
    f.write(f"Benign     : {benign_count:,} ({benign_count/total*100:.1f}%)\n")
    f.write(f"VUS        : {vus_count:,} ({vus_count/total*100:.1f}%)\n")
    if pathogenic_count > 0 and benign_count > 0:
        f.write(f"Imbalance  : {imbalance_ratio:.2f}:1\n")
        f.write(f"SMOTE      : {'Recommended' if imbalance_ratio > 5 else 'Not required'}\n\n")

    f.write("MISSING VALUES\n")
    f.write("-" * 40 + "\n")
    f.write(f"Columns with missing data: {len(missing_with_nulls)}\n")
    if len(missing_with_nulls) > 0:
        for _, row in missing_with_nulls.head(10).iterrows():
            f.write(f"  {row['column']:<50} {row['missing_pct']:.1f}%\n")
    f.write("\n")

    f.write("VISUALIZATIONS GENERATED\n")
    f.write("-" * 40 + "\n")
    for img in images_generated:
        f.write(f"  {img.name}\n")
    f.write("\n")

    f.write("OUTPUT FILES\n")
    f.write("-" * 40 + "\n")
    f.write(f"  images/  : {len(images_generated)} PNG files\n")
    f.write(f"  reports/ : pharmacogene_eda_report.txt\n")
    f.write(f"  metrics/ : missing_values.csv, correlation_matrix.csv, feature_statistics.csv\n")
    f.write("\n")

    f.write("NEXT STEPS\n")
    f.write("-" * 40 + "\n")
    f.write("  1. Note class imbalance ratio before training\n")
    f.write("  2. Remove leakage columns: clinical_significance_simple\n")
    f.write("  3. Apply correlation filter (r > 0.95)\n")
    f.write("  4. Proceed to 04_variant_impact_ml_features_eda.ipynb\n")

print(f"Saved: {report_path}")
print()
print("=" * 60)
print("PHARMACOGENE ML FEATURES EDA COMPLETE")
print("=" * 60)
print(f"  Images   : {len(images_generated)}")
print(f"  Reports  : 1")
print(f"  Metrics  : 3 CSV files")
print(f"  Output   : {BASE_OUT}")
print()
print("Next: 04_variant_impact_ml_features_eda.ipynb")